In [ ]:
import os
import xarray as xr
import numpy as np
import pandas as pd
from tqdm import tqdm

input_path = r"PATH_TO_SM_INPUT"


def load_sm_data(input_path):
    sm_files = sorted([f for f in os.listdir(input_path) if f.endswith('.nc')])
    sm_list = []
    for file in tqdm(sm_files, desc="Loading SM data"):
        file_path = os.path.join(input_path, file)
        ds = xr.open_dataset(file_path)
        ds['time'] = pd.to_datetime(ds.time.values).to_period('M').to_timestamp('M')
        sm_list.append(ds)

    sm_data = xr.concat(sm_list, dim='time').sortby('time')
    return sm_data


def create_sm_windows(sm_data, max_scale=48):
    sm_windows = {}
    for scale in tqdm(range(1, max_scale + 1), desc="Creating SM windows"):
        window_mean = (
            sm_data['SMrz']
            .rolling(time=scale, min_periods=1, center=False)
            .mean()
            .shift(time=1)
        )

        window_mean['time'] = sm_data['time']
        window_mean = window_mean.isel(time=slice(scale - 1, None))
        sm_windows[f"SM_{scale}"] = window_mean
    return sm_windows


def save_sm_windows(sm_windows, output_path):
    for scale, window in tqdm(sm_windows.items()):
        window = window.sel(time=slice('2001-01-31', '2020-12-31'))

        window['time'].encoding = {
            'units': 'days since 2000-01-01',
            'calendar': 'proleptic_gregorian'
        }

        output_file = os.path.join(output_path, f"{scale}_month_window.nc")
        window.to_netcdf(output_file)


def main():
    sm_data = load_sm_data(input_path)
    sm_windows = create_sm_windows(sm_data)
    save_sm_windows(sm_windows, r"PATH_TO_SM_WINDOWS_OUTPUT")


if __name__ == "__main__":
    main()


In [ ]:
import os
import xarray as xr
import re
from tqdm import tqdm

input_dir = r"PATH_TO_SM_WINDOWS_INPUT"
output_dir = r"PATH_TO_SM_MONTHLY_OUTPUT"
os.makedirs(output_dir, exist_ok=True)


def process_files():
    files = sorted([f for f in os.listdir(input_dir) if f.endswith('.nc')])

    for file in tqdm(files, desc="Processing scales"):
        match = re.match(r'SM_(\d+)_month_window\.nc', file)
        if not match:
            continue

        scale = match.group(1)
        file_path = os.path.join(input_dir, file)

        with xr.open_dataset(file_path) as ds:
            months = ds.time.dt.month

            for month in range(1, 13):
                monthly_data = ds.sel(time=months == month)

                if len(monthly_data.time) == 0:
                    continue

                output_file = f"SM_timescale{scale}_mon{month:02d}.nc"
                output_path = os.path.join(output_dir, output_file)

                monthly_data.time.encoding.update({
                    'units': 'days since 2000-01-01',
                    'calendar': 'proleptic_gregorian'
                })

                monthly_data.to_netcdf(output_path)


if __name__ == '__main__':
    process_files()
